In [1]:
"""
PHASE 4: COMPREHENSIVE EVALUATION & ANALYSIS (COMPLETE VERSION)
Updated to include: CF vs Content-Based vs Hybrid comparison

This phase provides publication-ready results:
1. Test set evaluation (CF vs Content vs Hybrid - all 3 embeddings)
2. Cold-start analysis
3. Statistical significance testing
4. Ablation study
5. Beyond-accuracy metrics
6. Publication-quality visualizations
"""

# ============================================================
# CELL 1: IMPORTS & SETUP
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats
import joblib
import warnings
import os
from collections import defaultdict
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity
import gc

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [2]:
print("="*70)
print("PHASE 4: COMPREHENSIVE EVALUATION & ANALYSIS")
print("="*70)

# ============================================================
# CELL 2: DEFINE HYBRID RECOMMENDER CLASS
# ============================================================

class HybridRecommender:
    def __init__(self, user_factors, item_factors, user_to_idx, item_to_idx,
                 item_similarity_dict, item_features, df_train,
                 user_bias, item_bias, global_mean, alpha=0.5):
        
        self.user_factors = user_factors
        self.item_factors = item_factors
        self.user_to_idx = user_to_idx
        self.item_to_idx = item_to_idx
        self.item_similarity_dict = item_similarity_dict
        self.item_features = item_features
        self.user_bias = user_bias
        self.item_bias = item_bias
        self.global_mean = global_mean
        self.alpha = alpha
        
        self.idx_to_item = {v: k for k, v in item_to_idx.items()}
        self.user_items = df_train.groupby('user_id')['item_id'].apply(set).to_dict()
        
        self.user_item_ratings = {}
        for uid in self.user_to_idx.keys():
            self.user_item_ratings[uid] = {}
        
        for _, row in df_train.iterrows():
            uid = row['user_id']
            iid = row['item_id']
            if iid in item_to_idx:
                self.user_item_ratings[uid][self.item_to_idx[iid]] = row['rating']
    
    def get_cf_score(self, user_id, item_id):
        """CF score with bias adjustment"""
        if user_id not in self.user_to_idx or item_id not in self.item_to_idx:
            return self.global_mean
        
        u_idx = self.user_to_idx[user_id]
        i_idx = self.item_to_idx[item_id]
        
        dot_product = np.dot(self.user_factors[u_idx], self.item_factors[i_idx])
        prediction = (self.global_mean + 
                     self.user_bias.get(user_id, 0.0) + 
                     self.item_bias.get(item_id, 0.0) + 
                     dot_product)
        
        return np.clip(prediction, 1.0, 5.0)
    
    def get_content_score(self, user_id, item_id):
        """Content-based score using similarities"""
        if user_id not in self.user_to_idx or item_id not in self.item_to_idx:
            return self.global_mean
        
        item_idx = self.item_to_idx[item_id]
        user_rated_items = self.user_items.get(user_id, set())
        
        if not user_rated_items or item_idx not in self.item_similarity_dict:
            return self.global_mean
        
        user_item_indices = {self.item_to_idx[iid]: iid 
                            for iid in user_rated_items 
                            if iid in self.item_to_idx}
        
        if not user_item_indices:
            return self.global_mean
        
        similar_data = self.item_similarity_dict[item_idx]
        similar_indices = similar_data['indices']
        similar_sims = similar_data['similarities'].astype(np.float32)
        
        weighted_sum = 0.0
        sim_sum = 0.0
        
        for sim_idx, sim_val in zip(similar_indices, similar_sims):
            if sim_idx in user_item_indices:
                rating = self.user_item_ratings[user_id].get(sim_idx, self.global_mean)
                weighted_sum += float(sim_val) * rating
                sim_sum += float(sim_val)
        
        if sim_sum == 0:
            return self.global_mean
        
        prediction = weighted_sum / sim_sum
        return np.clip(prediction, 1.0, 5.0)


PHASE 4: COMPREHENSIVE EVALUATION & ANALYSIS


In [3]:
# ============================================================
# CELL 3: LOAD ALL DATA AND MODELS
# ============================================================
print("\n[1/10] Loading models and data...")

# Load all 3 similarity dictionaries
sim_dict_w2v = joblib.load('item_similarity_w2v_500k.pkl')
sim_dict_bert = joblib.load('item_similarity_bert_500k.pkl')
sim_dict_both = joblib.load('item_similarity_both_500k.pkl')
print("✓ Loaded 3 similarity dictionaries")

# Load CF components
user_factors = joblib.load('user_factors_500k.pkl')
item_factors = joblib.load('item_factors_500k.pkl')
user_to_idx = joblib.load('user_to_idx_500k.pkl')
item_to_idx = joblib.load('item_to_idx_500k.pkl')
user_bias = joblib.load('user_bias_500k.pkl')
item_bias = joblib.load('item_bias_500k.pkl')
global_stats = joblib.load('global_stats_500k.pkl')
global_mean = global_stats['global_mean']
print("✓ Loaded CF components")

# Load validation results
validation_results = joblib.load('validation_results_TRUE.pkl')
embedding_comparison = joblib.load('embedding_comparison_500k.pkl')
print("✓ Loaded validation results")

# Load test set
df_test = pd.read_csv('test_data.csv')
df_test_filtered = df_test[
    (df_test['user_id'].isin(user_to_idx.keys())) & 
    (df_test['item_id'].isin(item_to_idx.keys()))
].copy()

print(f"✓ Test set loaded: {len(df_test_filtered):,} reviews")
print(f"  Coverage: {len(df_test_filtered)/len(df_test)*100:.1f}%")

# Load training data
df_train = pd.read_csv('train_data.csv')
df_train_sample = df_train.sample(n=500_000, random_state=42)


[1/10] Loading models and data...
✓ Loaded 3 similarity dictionaries
✓ Loaded CF components
✓ Loaded validation results
✓ Test set loaded: 109,363 reviews
  Coverage: 45.6%


In [4]:
# ============================================================
# CELL 4: COMPLETE EVALUATION (CF vs Content vs Hybrid)
# ============================================================

print("\n[2/10] Complete evaluation: CF vs Content-Based vs Hybrid...")

# Sample test data
test_sample = df_test_filtered.sample(n=min(20000, len(df_test_filtered)), random_state=42)
actual_test = test_sample['rating'].tolist()

print(f"  Evaluating {len(test_sample):,} test samples")

# Build lookups
user_items_lookup = df_train_sample.groupby('user_id')['item_id'].apply(set).to_dict()
user_item_ratings = {}
for uid in user_to_idx.keys():
    user_item_ratings[uid] = {}
for _, row in df_train_sample.iterrows():
    uid, iid = row['user_id'], row['item_id']
    if iid in item_to_idx:
        user_item_ratings[uid][item_to_idx[iid]] = row['rating']

# Helper functions
def get_cf_score(user_id, item_id):
    """Pure CF prediction"""
    if user_id not in user_to_idx or item_id not in item_to_idx:
        return global_mean
    u_idx, i_idx = user_to_idx[user_id], item_to_idx[item_id]
    dot = np.dot(user_factors[u_idx], item_factors[i_idx])
    pred = global_mean + user_bias.get(user_id, 0.0) + item_bias.get(item_id, 0.0) + dot
    return np.clip(pred, 1.0, 5.0)

def get_content_score(user_id, item_id, sim_dict):
    """Pure content-based prediction"""
    if user_id not in user_to_idx or item_id not in item_to_idx:
        return global_mean
    
    item_idx = item_to_idx[item_id]
    user_rated = user_items_lookup.get(user_id, set())
    
    if not user_rated or item_idx not in sim_dict:
        return global_mean
    
    user_item_indices = {item_to_idx[iid]: iid for iid in user_rated if iid in item_to_idx}
    if not user_item_indices:
        return global_mean
    
    similar_data = sim_dict[item_idx]
    weighted_sum, sim_sum = 0.0, 0.0
    
    for sim_idx, sim_val in zip(similar_data['indices'], similar_data['similarities']):
        if sim_idx in user_item_indices:
            rating = user_item_ratings[user_id].get(sim_idx, global_mean)
            weighted_sum += float(sim_val) * rating
            sim_sum += float(sim_val)
    
    return np.clip(weighted_sum / sim_sum, 1.0, 5.0) if sim_sum > 0 else global_mean

# Store all predictions
all_predictions = {}

# 1. CF Only (baseline)
print("\n  [CF Only]")
cf_preds = [get_cf_score(row['user_id'], row['item_id']) 
            for _, row in test_sample.iterrows()]
all_predictions['CF Only'] = cf_preds

# 2-4. Content-Based Only (3 embeddings)
embeddings = {
    'Content (Word2Vec)': sim_dict_w2v,
    'Content (BERT)': sim_dict_bert,
    'Content (Combined)': sim_dict_both
}

for name, sim_dict in embeddings.items():
    print(f"  [{name}]")
    content_preds = [get_content_score(row['user_id'], row['item_id'], sim_dict) 
                     for _, row in test_sample.iterrows()]
    all_predictions[name] = content_preds

# 5-7. Hybrid (3 embeddings)
hybrid_configs = {
    'Hybrid (Word2Vec)': ('Content (Word2Vec)', 0.6),
    'Hybrid (BERT)': ('Content (BERT)', 0.6),
    'Hybrid (Combined)': ('Content (Combined)', 0.6)
}

for hybrid_name, (content_name, alpha) in hybrid_configs.items():
    print(f"  [{hybrid_name}]")
    content_preds = all_predictions[content_name]
    hybrid_preds = [alpha * cf + (1-alpha) * cont 
                    for cf, cont in zip(cf_preds, content_preds)]
    all_predictions[hybrid_name] = hybrid_preds

# Compute metrics
print("\nComputing metrics...")
results_complete = {}
for method, preds in all_predictions.items():
    rmse = np.sqrt(mean_squared_error(actual_test, preds))
    mae = mean_absolute_error(actual_test, preds)
    results_complete[method] = {'rmse': rmse, 'mae': mae}

# Display results
print("\n" + "="*70)
print("COMPLETE RESULTS: CF vs Content-Based vs Hybrid")
print("="*70)
print("\nMethod                    | RMSE   | MAE    | vs CF")
print("--------------------------|--------|--------|--------")

baseline_rmse = results_complete['CF Only']['rmse']

method_order = [
    'CF Only',
    'Content (Word2Vec)',
    'Content (BERT)', 
    'Content (Combined)',
    'Hybrid (Word2Vec)',
    'Hybrid (BERT)',
    'Hybrid (Combined)'
]

for method in method_order:
    if method in results_complete:
        rmse = results_complete[method]['rmse']
        mae = results_complete[method]['mae']
        improvement = ((baseline_rmse - rmse) / baseline_rmse) * 100
        print(f"{method:25} | {rmse:.4f} | {mae:.4f} | {improvement:+.1f}%")

# Key insights
print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

best_cf = results_complete['CF Only']['rmse']
best_content = min([results_complete[m]['rmse'] for m in embeddings.keys()])
best_hybrid = min([results_complete[m]['rmse'] for m in hybrid_configs.keys()])

print(f"\n📊 Performance by Approach:")
print(f"  CF Only:           {best_cf:.4f}")
print(f"  Best Content-Based: {best_content:.4f} ({((best_cf-best_content)/best_cf*100):+.1f}% vs CF)")
print(f"  Best Hybrid:        {best_hybrid:.4f} ({((best_cf-best_hybrid)/best_cf*100):+.1f}% vs CF)")

content_w2v_rmse = results_complete['Content (Word2Vec)']['rmse']
hybrid_w2v_rmse = results_complete['Hybrid (Word2Vec)']['rmse']
improvement_over_content = ((content_w2v_rmse - hybrid_w2v_rmse) / content_w2v_rmse * 100)
improvement_over_cf = ((best_cf - hybrid_w2v_rmse) / best_cf * 100)

print(f"\n🔀 Hybrid (Word2Vec) combines strengths:")
print(f"  vs CF:           {improvement_over_cf:+.1f}%")
print(f"  vs Content-Only: {improvement_over_content:+.1f}%")

best_overall = min(results_complete.items(), key=lambda x: x[1]['rmse'])
print(f"\n🏆 Overall Best: {best_overall[0]} (RMSE = {best_overall[1]['rmse']:.4f})")

# Store for later cells
test_results = {
    'cf': results_complete['CF Only'],
    'word2vec': results_complete['Hybrid (Word2Vec)'],
    'bert': results_complete['Hybrid (BERT)'],
    'combined': results_complete['Hybrid (Combined)']
}

all_test_preds = {
    'cf': cf_preds,
    'word2vec': all_predictions['Hybrid (Word2Vec)'],
    'bert': all_predictions['Hybrid (BERT)'],
    'combined': all_predictions['Hybrid (Combined)']
}

best_test = min([(k, v['rmse']) for k, v in test_results.items() if k != 'cf'], key=lambda x: x[1])


[2/10] Complete evaluation: CF vs Content-Based vs Hybrid...
  Evaluating 20,000 test samples

  [CF Only]
  [Content (Word2Vec)]
  [Content (BERT)]
  [Content (Combined)]
  [Hybrid (Word2Vec)]
  [Hybrid (BERT)]
  [Hybrid (Combined)]

Computing metrics...

COMPLETE RESULTS: CF vs Content-Based vs Hybrid

Method                    | RMSE   | MAE    | vs CF
--------------------------|--------|--------|--------
CF Only                   | 0.9726 | 0.5968 | +0.0%
Content (Word2Vec)        | 1.1346 | 0.8701 | -16.7%
Content (BERT)            | 1.1363 | 0.8725 | -16.8%
Content (Combined)        | 1.1366 | 0.8727 | -16.9%
Hybrid (Word2Vec)         | 0.9104 | 0.6737 | +6.4%
Hybrid (BERT)             | 0.9117 | 0.6746 | +6.3%
Hybrid (Combined)         | 0.9118 | 0.6746 | +6.3%

KEY INSIGHTS

📊 Performance by Approach:
  CF Only:           0.9726
  Best Content-Based: 1.1346 (-16.7% vs CF)
  Best Hybrid:        0.9104 (+6.4% vs CF)

🔀 Hybrid (Word2Vec) combines strengths:
  vs CF:           +6.

In [5]:
# ============================================================
# CELL 5: COLD-START ANALYSIS (ALL METHODS)
# ============================================================

print("\n[3/10] Cold-start analysis (all approaches)...")

item_rating_counts = df_train_sample.groupby('item_id').size().to_dict()
test_sample['item_popularity'] = test_sample['item_id'].map(lambda x: item_rating_counts.get(x, 0))

cold_categories = [
    ('Cold (1-5)', 1, 5),
    ('Warm (6-20)', 6, 20),
    ('Popular (21+)', 21, 10000)
]

print("\n" + "="*70)
print("COLD-START COMPARISON (All Methods)")
print("="*70)
print("\nCategory    | CF     | Cont-W2V | Cont-BERT | Hyb-W2V | Hyb-BERT | Winner")
print("------------|--------|----------|-----------|---------|----------|--------")

cold_start_complete = {}

for cat_name, min_pop, max_pop in cold_categories:
    mask = (test_sample['item_popularity'] >= min_pop) & (test_sample['item_popularity'] <= max_pop)
    if mask.sum() == 0:
        continue
    
    indices = test_sample[mask].index
    seg_actual = [actual_test[i] for i, idx in enumerate(test_sample.index) if idx in indices]
    
    seg_rmses = {}
    for method_name in ['CF Only', 'Content (Word2Vec)', 'Content (BERT)', 
                        'Hybrid (Word2Vec)', 'Hybrid (BERT)']:
        seg_preds = [all_predictions[method_name][i] 
                     for i, idx in enumerate(test_sample.index) if idx in indices]
        seg_rmses[method_name] = np.sqrt(mean_squared_error(seg_actual, seg_preds))
    
    winner = min(seg_rmses.items(), key=lambda x: x[1])[0]
    
    cold_start_complete[cat_name] = seg_rmses
    
    cf_rmse = seg_rmses['CF Only']
    cont_w2v = seg_rmses['Content (Word2Vec)']
    cont_bert = seg_rmses['Content (BERT)']
    hyb_w2v = seg_rmses['Hybrid (Word2Vec)']
    hyb_bert = seg_rmses['Hybrid (BERT)']
    
    print(f"{cat_name:11} | {cf_rmse:.4f} | {cont_w2v:.4f}   | {cont_bert:.4f}    | {hyb_w2v:.4f}  | {hyb_bert:.4f}   | {winner.split('(')[0].strip()}")

# Find cold-start improvement
if 'Cold (1-5)' in cold_start_complete:
    cold_cf = cold_start_complete['Cold (1-5)']['CF Only']
    cold_best = min([cold_start_complete['Cold (1-5)'][m] 
                     for m in ['Content (Word2Vec)', 'Content (BERT)', 
                               'Hybrid (Word2Vec)', 'Hybrid (BERT)']])
    cold_improvement = ((cold_cf - cold_best) / cold_cf) * 100
    print(f"\n✓ Best cold-start improvement: {cold_improvement:.1f}%")


[3/10] Cold-start analysis (all approaches)...

COLD-START COMPARISON (All Methods)

Category    | CF     | Cont-W2V | Cont-BERT | Hyb-W2V | Hyb-BERT | Winner
------------|--------|----------|-----------|---------|----------|--------
Cold (1-5)  | 1.1841 | 1.1147   | 1.1146    | 1.0229  | 1.0253   | Hybrid
Warm (6-20) | 1.0382 | 1.1150   | 1.1137    | 0.9498  | 0.9501   | Hybrid
Popular (21+) | 0.8521 | 1.1491   | 1.1525    | 0.8487  | 0.8497   | Hybrid

✓ Best cold-start improvement: 13.6%


In [6]:
# ============================================================
# CELL 6: STATISTICAL SIGNIFICANCE TEST
# ============================================================

print("\n[4/10] Statistical significance testing...")

best_embedding_name = best_test[0].replace('_', ' ').title()
best_hybrid_preds = all_test_preds[best_test[0]]

cf_errors = np.array(actual_test) - np.array(cf_preds)
hybrid_errors = np.array(actual_test) - np.array(best_hybrid_preds)
cf_sq_errors = cf_errors ** 2
hybrid_sq_errors = hybrid_errors ** 2

t_stat, p_value = stats.ttest_rel(cf_sq_errors, hybrid_sq_errors)

print("\n" + "="*70)
print("STATISTICAL SIGNIFICANCE TEST")
print("="*70)
print(f"\nPaired t-test ({best_embedding_name} vs CF on test set):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.6f}")

if p_value < 0.001:
    print(f"  Result: *** HIGHLY SIGNIFICANT (p < 0.001)")
elif p_value < 0.01:
    print(f"  Result: ** VERY SIGNIFICANT (p < 0.01)")
elif p_value < 0.05:
    print(f"  Result: * SIGNIFICANT (p < 0.05)")
else:
    print(f"  Result: NOT SIGNIFICANT (p >= 0.05)")

mean_diff = np.mean(cf_sq_errors - hybrid_sq_errors)
pooled_std = np.sqrt((np.var(cf_sq_errors) + np.var(hybrid_sq_errors)) / 2)
cohens_d = mean_diff / pooled_std
print(f"\n  Effect size (Cohen's d): {cohens_d:.4f}")


[4/10] Statistical significance testing...

STATISTICAL SIGNIFICANCE TEST

Paired t-test (Word2Vec vs CF on test set):
  t-statistic: 15.4959
  p-value: 0.000000
  Result: *** HIGHLY SIGNIFICANT (p < 0.001)

  Effect size (Cohen's d): 0.0592


In [7]:
# ============================================================
# CELL 7: VISUALIZATION
# ============================================================

print("\n[5/10] Creating visualizations...")

os.makedirs('figuresv3', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Overall Performance Comparison
ax = axes[0, 0]
methods_viz = ['CF Only', 'Content\n(Word2Vec)', 'Content\n(BERT)', 
               'Hybrid\n(Word2Vec)', 'Hybrid\n(BERT)']
rmses_viz = [results_complete[m.replace('\n', ' ')]['rmse'] for m in methods_viz]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
bars = ax.bar(range(len(methods_viz)), rmses_viz, color=colors, alpha=0.8)
ax.set_ylabel('RMSE', fontsize=14)
ax.set_title('(A) Overall Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(methods_viz)))
ax.set_xticklabels(methods_viz, fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar, rmse in zip(bars, rmses_viz):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{rmse:.3f}', ha='center', va='bottom', fontsize=10)

# Panel 2: Cold-Start Performance
ax = axes[0, 1]
categories = list(cold_start_complete.keys())
x = np.arange(len(categories))
width = 0.15

cf_rmses = [cold_start_complete[cat]['CF Only'] for cat in categories]
cont_rmses = [cold_start_complete[cat]['Content (Word2Vec)'] for cat in categories]
hyb_rmses = [cold_start_complete[cat]['Hybrid (Word2Vec)'] for cat in categories]

ax.bar(x - width, cf_rmses, width, label='CF Only', alpha=0.8)
ax.bar(x, cont_rmses, width, label='Content', alpha=0.8)
ax.bar(x + width, hyb_rmses, width, label='Hybrid', alpha=0.8)

ax.set_ylabel('RMSE', fontsize=14)
ax.set_title('(B) Cold-Start Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([cat.split('(')[0].strip() for cat in categories])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Panel 3: Improvement Summary
ax = axes[1, 0]
improvements = {
    'Content\nvs CF': ((baseline_rmse - best_content) / baseline_rmse * 100),
    'Hybrid\nvs CF': ((baseline_rmse - best_hybrid) / baseline_rmse * 100),
    'Hybrid\nvs Content': ((best_content - best_hybrid) / best_content * 100)
}

bars = ax.bar(improvements.keys(), improvements.values(), 
              color=['#ff7f0e', '#2ca02c', '#9467bd'], alpha=0.8)
ax.set_ylabel('Improvement (%)', fontsize=14)
ax.set_title('(C) Relative Improvements', fontsize=14, fontweight='bold')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)

for bar, (name, val) in zip(bars, improvements.items()):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:+.1f}%', ha='center', va='bottom' if val > 0 else 'top', 
            fontsize=12, fontweight='bold')

# Panel 4: Embedding Comparison
ax = axes[1, 1]
embed_methods = ['Word2Vec', 'BERT', 'Combined']
content_rmses = [results_complete[f'Content ({m})']['rmse'] for m in embed_methods]
hybrid_rmses = [results_complete[f'Hybrid ({m})']['rmse'] for m in embed_methods]

x = np.arange(len(embed_methods))
width = 0.35

ax.bar(x - width/2, content_rmses, width, label='Content-Based', alpha=0.8)
ax.bar(x + width/2, hybrid_rmses, width, label='Hybrid', alpha=0.8)

ax.set_ylabel('RMSE', fontsize=14)
ax.set_title('(D) Embedding Method Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(embed_methods)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figuresv3/fig11_complete_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("✓ Saved: figuresv3/fig11_complete_comparison.png")


[5/10] Creating visualizations...
✓ Saved: figuresv3/fig11_complete_comparison.png


In [8]:
# ============================================================
# CELL 8: SAVE RESULTS
# ============================================================

print("\n[6/10] Saving comprehensive results...")

complete_results = {
    'predictions': all_predictions,
    'metrics': results_complete,
    'cold_start': cold_start_complete,
    'test_results': test_results,
    'statistical_test': {
        't_statistic': t_stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'significant': p_value < 0.05
    },
    'insights': {
        'best_cf': best_cf,
        'best_content': best_content,
        'best_hybrid': best_hybrid,
        'overall_best': best_overall[0]
    }
}

joblib.dump(complete_results, 'phase4_complete_results.pkl')
print("✓ Complete results saved to 'phase4_complete_results.pkl'")


[6/10] Saving comprehensive results...
✓ Complete results saved to 'phase4_complete_results.pkl'


In [9]:
# ============================================================
# CELL 9: PAPER-READY SUMMARY
# ============================================================

print("\n[7/10] Generating paper summary...")

print("\n" + "="*70)
print("PAPER-READY SUMMARY")
print("="*70)

print("\n📊 MAIN RESULTS:")
print(f"\n  Test Set Performance:")
print(f"    CF Baseline:          RMSE = {best_cf:.4f}")
print(f"    Best Content-Based:   RMSE = {best_content:.4f} ({((best_cf-best_content)/best_cf*100):+.1f}%)")
print(f"    Best Hybrid:          RMSE = {best_hybrid:.4f} ({((best_cf-best_hybrid)/best_cf*100):+.1f}%)")
print(f"    Statistical Significance: p = {p_value:.6f} {'***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'n.s.'}")

if 'Cold (1-5)' in cold_start_complete:
    print(f"\n  Cold-Start Items (1-5 ratings):")
    print(f"    CF:              {cold_start_complete['Cold (1-5)']['CF Only']:.4f}")
    print(f"    Content (W2V):   {cold_start_complete['Cold (1-5)']['Content (Word2Vec)']:.4f}")
    print(f"    Hybrid (W2V):    {cold_start_complete['Cold (1-5)']['Hybrid (Word2Vec)']:.4f}")
    print(f"    Best improvement: {cold_improvement:.1f}%")

print("\n" + "="*70)
print("PHASE 4 COMPLETE!")
print("="*70)
print(f"\n🎯 Key Takeaways:")
print(f"  • CF baseline:      {best_cf:.4f}")
print(f"  • Best content:     {best_content:.4f} ({((best_cf-best_content)/best_cf*100):+.1f}%)")
print(f"  • Best hybrid:      {best_hybrid:.4f} ({((best_cf-best_hybrid)/best_cf*100):+.1f}%)")
print(f"  • Hybrid combines strengths of both CF and content-based!")


[7/10] Generating paper summary...

PAPER-READY SUMMARY

📊 MAIN RESULTS:

  Test Set Performance:
    CF Baseline:          RMSE = 0.9726
    Best Content-Based:   RMSE = 1.1346 (-16.7%)
    Best Hybrid:          RMSE = 0.9104 (+6.4%)
    Statistical Significance: p = 0.000000 ***

  Cold-Start Items (1-5 ratings):
    CF:              1.1841
    Content (W2V):   1.1147
    Hybrid (W2V):    1.0229
    Best improvement: 13.6%

PHASE 4 COMPLETE!

🎯 Key Takeaways:
  • CF baseline:      0.9726
  • Best content:     1.1346 (-16.7%)
  • Best hybrid:      0.9104 (+6.4%)
  • Hybrid combines strengths of both CF and content-based!
